# Section 3: Robustness — Lee (2008) (28 pts)

ECON 5200 · Replication of Lee (2008), *Randomized Experiments from Non-random Selection in U.S. House Elections*.

**Study design.** The outcome is next-election Democratic two-party vote share, and the running variable is previous-election Democratic two-party vote share minus 0.5. The cutoff is zero. This file uses the assigned **Lee, Moretti & Butler (2004)** data (1948–1990), **not** Lee's original 1946–1998 sample.

This notebook is **standalone** (can run in Colab) **and repository-friendly** (uses `data/lmb-data.dta` or `download_data.py` if available). It writes figures and CSVs to `output/`. It recomputes the Section 2 benchmark rather than hard-coding the previously reported 0.0861. It carries out four exercises that go beyond the published headline: (1) bandwidth; (2) polynomial order; (3) donut RD; (4) placebo cutoffs.

**AI disclosure.** Record AI assistance honestly in `ai-appendix.md` according to the instructor's latest policy. Be able to explain the models and limitations in your own words.


## Setup — packages, paths, and reproducible output


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display, Markdown

CWD = Path.cwd().resolve()
if (CWD / 'download_data.py').exists() or (CWD / 'data').exists():
    ROOT = CWD
elif (CWD.parent / 'download_data.py').exists():
    ROOT = CWD.parent
else:
    ROOT = CWD
sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / 'output'
OUTPUT.mkdir(parents=True, exist_ok=True)

PAPER_JUMP = 0.077          # Lee (2008), Table 2, column (1); share units
BASELINE_H = 0.5           # full feasible two-party vote-share margin range
BASELINE_ORDER = 4         # quartic fitted separately on both sides
print('Project root:', ROOT)
print('Output directory:', OUTPUT)


## Data and common estimation method

We first look for `data/lmb-data.dta` in the repository. If it is missing, try the template's `download_data.py` helper; otherwise, download from the **public source specified in the course brief**. This keeps the notebook usable even when opened alone in Colab. Downloading requires internet access the first time.

**Method for the four exercises.** Weighted least squares on observations within bandwidth `h`, with separate left/right polynomials and an intercept jump at the specified cutoff. The default kernel is triangular for local fits; standard errors are clustered by `state × district × decade`, to align with the baseline implementation in Section 2. Normal-approximation 95% intervals are reported. For the original comparison, the global quartic is re-estimated with uniform weights.

*Interpretation warning:* An estimate that remains positive is not automatically statistically significant. Placebo effects and donut estimates are specification diagnostics, not formal proof of RD validity.


In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/scunning1975/mixtape/master/lmb-data.dta'
DATA_FILE = ROOT / 'data' / 'lmb-data.dta'
DATA_FILE.parent.mkdir(parents=True, exist_ok=True)

if DATA_FILE.exists():
    source = DATA_FILE
else:
    source = None
    if (ROOT / 'download_data.py').exists():
        try:
            from download_data import fetch
            possible_paths = fetch('lee')
            source = Path(possible_paths[0])
        except Exception as exc:
            print('download_data.py unavailable for this run:', exc)
    if source is None:
        try:
            import urllib.request
            urllib.request.urlretrieve(DATA_URL, DATA_FILE)
            source = DATA_FILE
        except Exception as exc:
            raise RuntimeError('Could not obtain the data. Place lmb-data.dta in data/ or enable internet access.') from exc

raw = pd.read_stata(source)
needed = ['demvoteshare', 'lagdemvoteshare', 'year', 'state', 'district']
missing = sorted(set(needed) - set(raw.columns))
if missing:
    raise KeyError(f'Dataset lacks required columns: {missing}')

lmb = raw.dropna(subset=needed).copy()
lmb['margin'] = lmb['lagdemvoteshare'].astype(float) - 0.5
lmb['win'] = (lmb['margin'] >= 0).astype(int)
lmb['decade'] = (lmb['year'].astype(int) // 10) * 10
lmb['cluster'] = pd.factorize(
    lmb[['state', 'district', 'decade']].astype(str).agg('|'.join, axis=1)
)[0]

# Reproduce Section 2 global quartic, uniform-weight benchmark.
# 'tau' is the discontinuity coefficient in share units.
def rd_estimate(data, h, order=1, cutoff=0.0, kernel='triangular', donut=0.0,
                outcome='demvoteshare', label=''):
    if h <= 0 or donut < 0 or donut >= h:
        raise ValueError('Require 0 <= donut < bandwidth.')
    if kernel not in ('triangular', 'uniform'):
        raise ValueError('kernel must be triangular or uniform')
    d = data.dropna(subset=[outcome, 'margin', 'cluster']).copy()
    x = d['margin'].astype(float) - cutoff
    keep = (x.abs() <= h) & (x.abs() > donut)
    d = d.loc[keep].copy()
    x = x.loc[keep]
    right = (x >= 0).astype(float)
    nl, nr = int((right == 0).sum()), int((right == 1).sum())
    if min(nl, nr) < max(20, 3*(order+1)):
        raise ValueError(f'Not enough observations on both sides: n_left={nl}, n_right={nr}')
    X = pd.DataFrame({'const': 1., 'right': right}, index=d.index)
    for j in range(1, order+1):
        X[f'x{j}'] = x**j
        X[f'right_x{j}'] = right * (x**j)
    weights = (1 - x.abs()/h).clip(lower=1e-8) if kernel == 'triangular' else np.ones(len(d))
    fit = sm.WLS(d[outcome].astype(float), X, weights=weights).fit(
        cov_type='cluster', cov_kwds={'groups': d['cluster'].to_numpy()}
    )
    b, se = float(fit.params['right']), float(fit.bse['right'])
    pval = float(fit.pvalues['right'])
    return {'specification': label, 'estimate': b, 'se': se, 'ci_low': b-1.96*se,
            'ci_high': b+1.96*se, 'p_value': pval, 'n': int(fit.nobs),
            'n_left': nl, 'n_right': nr, 'clusters': int(d['cluster'].nunique()),
            'h': h, 'order': order, 'kernel': kernel, 'donut': donut, 'cutoff': cutoff}

BASELINE = rd_estimate(lmb, BASELINE_H, order=BASELINE_ORDER,
                       kernel='uniform', label='Section 2: global quartic')
print(f"Dataset: {len(raw):,} raw rows, {len(lmb):,} complete rows, {lmb['year'].min():.0f}–{lmb['year'].max():.0f}")
print(f"Section 2 benchmark: {100*BASELINE['estimate']:.2f} pp "
      f"(clustered SE {100*BASELINE['se']:.2f} pp; N={BASELINE['n']:,})")
print(f"Lee (2008) published benchmark: {100*PAPER_JUMP:.2f} pp")

all_results = []

def record(rows, filename):
    frame = pd.DataFrame(rows)
    frame.to_csv(OUTPUT / filename, index=False)
    all_results.extend(rows)
    cols = ['specification','estimate','se','ci_low','ci_high','p_value','n']
    display(frame[cols].round(4))
    return frame

def coefficient_plot(frame, title, filename):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    xs = np.arange(len(frame))
    estimates = frame['estimate'].to_numpy(dtype=float)*100
    errs = 1.96*frame['se'].to_numpy(dtype=float)*100
    ax.errorbar(xs, estimates, yerr=errs, fmt='o', capsize=4)
    ax.axhline(0, linewidth=1, linestyle=':')
    ax.axhline(100*BASELINE['estimate'], linestyle='--', label='S2 global quartic')
    ax.axhline(100*PAPER_JUMP, linestyle=':', label='Lee (2008): 7.7 pp')
    ax.set_xticks(xs, frame['specification'], rotation=20, ha='right')
    ax.set_ylabel('Discontinuity (percentage points); 95% CI')
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT / filename, dpi=160)
    plt.show()
    plt.close(fig)

# Interpretations below are automatically generated from estimates, not guessed.
def significance_phrase(row):
    return ('the 95% interval excludes zero' if row['ci_low']>0 or row['ci_high']<0
            else 'the 95% interval includes zero')


## Four exercises (four independently justified changes)


### Exercise 1 — Bandwidth sensitivity

**What I did.** I estimated local-linear RD models with triangular weights at four predetermined bandwidths (0.025, 0.05, 0.10, and 0.20 in two-party vote-share units), holding the polynomial order and inference method fixed. Narrow windows focus more closely on close elections but have fewer observations; wider windows gain precision at the risk of functional-form bias. I compare these estimates with the Section 2 full-range quartic and Lee's published 7.7 pp. These bandwidths are a **sensitivity grid**, not claims to be MSE-optimal.


In [ ]:
bandwidths = [0.025, 0.05, 0.10, 0.20]
bw_rows = [rd_estimate(lmb, h, order=1, kernel='triangular',
                       label=f'h = {h:g}') for h in bandwidths]
bw = record(bw_rows, 'robustness_bandwidth.csv')
coefficient_plot(bw, 'Exercise 1: bandwidth sensitivity (local linear)', 'robustness_bandwidth.png')
print('Baseline full-range quartic:', f"{100*BASELINE['estimate']:.2f} pp")

min_b, max_b = 100*bw['estimate'].min(), 100*bw['estimate'].max()
strong = int((bw['ci_low'] > 0).sum())
display(Markdown(
    f"**What I found.** Across these four bandwidths, the jump ranges from **{min_b:.2f} to {max_b:.2f} percentage points**; "
    f"**{strong}/4** of the 95% CIs are entirely above zero. "
    f"\n\n**What it implies.** " +
    ("The estimated positive effect survives these bandwidth choices with conventional 95% precision. " if strong==4 else
     "The sign or precision is not equally secure across bandwidth choices. ") +
    "A change in the point estimate when the window narrows indicates sensitivity to the choice of comparison elections."
))


**What I found and what it implies.** The code Cell above prints the estimates, clustered standard errors, sample sizes and intervals; the cell below generates an interpretation **after the real data run**. Review the result rather than claiming that all bandwidths confirm the paper.


### Exercise 2 — Functional-form sensitivity

**What I did.** I compare local linear, quadratic, and cubic fits using the *same* 0.10 bandwidth and triangular kernel, so polynomial order is the only change. This tests whether curvature near the threshold changes the jump. Higher-order polynomial estimates can be unstable even in a local window; they are not automatically preferred.


In [ ]:
order_rows = [rd_estimate(lmb, 0.10, order=p, kernel='triangular',
                          label=f'Polynomial order {p}') for p in (1,2,3)]
poly = record(order_rows, 'robustness_polynomial.csv')
coefficient_plot(poly, 'Exercise 2: polynomial order at h = 0.10', 'robustness_polynomial.png')
spread = 100*(poly['estimate'].max()-poly['estimate'].min())
count = int((poly['ci_low'] > 0).sum())
display(Markdown(
    f"**What I found.** At h=0.10, the spread between the largest and smallest point estimate is **{spread:.2f} percentage points**. "
    f"**{count}/3** intervals exclude zero on the positive side. "
    "\n\n**What it implies.** " +
    ("The result retains its positive direction and 95% precision across these three local polynomial orders." if count==3 else
     "Statistical precision depends on the polynomial order, so a single functional form is not decisive.")
))


**What I found and what it implies.** The executed Cell 11 reports the realized range and inference. In the final report, emphasize any meaningful changes in size and confidence intervals rather than simply listing models.


### Exercise 3 — Donut RD (possible problems near extremely close races)

**What I did.** I keep a local-linear triangular-kernel RD with outer bandwidth 0.10, but remove observations with absolute previous-election margin within 0.005 or 0.010 of zero. The first row uses no donut. This is a diagnostic for unusual behavior in extremely close elections (including potential precise sorting or measurement issues). A donut changes the estimand/extrapolation and cannot by itself establish random assignment.


In [ ]:
donut_rows = [rd_estimate(lmb, 0.10, order=1, kernel='triangular', donut=d,
                          label=f'Donut radius {d:g}') for d in (0, 0.005, 0.010)]
donut = record(donut_rows, 'robustness_donut.csv')
coefficient_plot(donut, 'Exercise 3: donut RD (h = 0.10)', 'robustness_donut.png')
ref = float(donut.iloc[0]['estimate'])
maxshift = 100*max(abs(float(r)-ref) for r in donut['estimate'])
count = int((donut['ci_low'] > 0).sum())
display(Markdown(
    f"**What I found.** Removing observations near zero changes the point estimate by at most **{maxshift:.2f} pp** "
    f"relative to the no-donut local fit; **{count}/3** intervals are entirely positive. "
    "\n\n**What it implies.** " +
    ("The main local result is not dependent on including the very closest races in these checks." if count==3 else
     "Inference becomes less secure in at least one donut specification, so very close races or data loss may matter.") +
    " This check does not establish whether manipulation occurred."
))


**What I found and what it implies.** Cell 14 gives the estimated change and interval for each donut choice. In particular, consider the increased extrapolation distance when the nearest races are excluded.


### Exercise 4 — Placebo cutoffs away from the real election threshold

**What I did.** I test artificial cutoffs at −0.10 *among prior losers only* and +0.10 *among prior winners only*. In either subgroup, the Democratic victory treatment does not change at the placebo cutoff. I use local-linear triangular fits with bandwidth 0.05 to keep each placebo comparison on one side of the **real** zero cutoff. Significant placebo discontinuities would suggest the estimator can find jumps where the treatment rule does not change. Failure to reject a placebo jump does **not** prove the real RD assumption.


In [ ]:
placebo_rows = []
for cutoff, subset, side in [(-0.10, lmb[lmb['margin']<0], 'losers only'),
                             (0.10, lmb[lmb['margin']>0], 'winners only')]:
    placebo_rows.append(rd_estimate(
        subset, 0.05, order=1, cutoff=cutoff, kernel='triangular',
        label=f'Cutoff {cutoff:+.2f} ({side})'
    ))
placebo = record(placebo_rows, 'robustness_placebo.csv')
coefficient_plot(placebo, 'Exercise 4: placebo cutoffs', 'robustness_placebo.png')
false_positive_count = int((placebo['p_value'] < 0.05).sum())
display(Markdown(
    f"**What I found.** **{false_positive_count}/2** placebo cutoffs have cluster-robust p < 0.05. "
    "\n\n**What it implies.** " +
    ("Neither placebo jump is statistically distinguishable from zero at 5%; this supports, but does not prove, a localized threshold interpretation." if false_positive_count==0 else
     "At least one placebo cutoff shows a statistically detectable jump despite no change in treatment; this cautions against taking the main estimated discontinuity as automatically causal.")
))

# Export a single machine-readable table for the Section 3 report and clean checkout.
all_table = pd.DataFrame(all_results)
all_table.insert(0, 'exercise',
                 ['bandwidth']*len(bw_rows)+['polynomial']*len(order_rows)+
                 ['donut']*len(donut_rows)+['placebo']*len(placebo_rows))
all_table.to_csv(OUTPUT / 'robustness_summary.csv', index=False)
print('Saved', OUTPUT / 'robustness_summary.csv')
print('Report units: multiply share estimates and SEs by 100 for percentage points.')


**What I found and what it implies.** The executed Cell 17 reports results for both placebo cutoffs. A non-significant placebo is only a diagnostic, not evidence that all identifying assumptions hold.

---

### Overall assessment to take into Section 4

Read `output/robustness_summary.csv` and the four figures, then answer: (a) is the jump generally positive? (b) under which choices does its confidence interval include zero? (c) do placebo cutoffs suggest spurious discontinuities? (d) how does the size compare with the published **7.7 pp** and the *re-estimated* Section 2 assigned-data benchmark? Do not claim the difference from 7.7 pp has been causally attributed to the missing years; this file lacks those elections.

**Generated deliverables:** `output/robustness_bandwidth.csv/.png`, `robustness_polynomial.csv/.png`, `robustness_donut.csv/.png`, `robustness_placebo.csv/.png`, and `robustness_summary.csv`. The figures include 95% cluster-robust normal-approximation intervals. In the final report each exercise should include a concise `What I did / What I found / What it implies` paragraph, with actual output numbers.
